**Data Info**

train.csv
* ID : 사건 샘플 ID
* first_party : 사건의 첫 번째 당사자
* second_party : 사건의 두 번째 당사자
* facts : 사건 내용
* first_party_winner : 첫 번째 당사자의 승소 여부 (0 : 패배, 1 : 승리)

test.csv
* ID : 사건 샘플 ID
* first_party : 사건의 첫 번째 당사자
* second_party : 사건의 두 번째 당사자
* facts : 사건 내용

sample_submission.csv - 제출 양식
* ID : 사건 샘플 ID
* first_party_winner : 예측한 첫 번째 당사자의 승소 여부 (0 : 패배, 1 : 승리)

# Baseline: TF-IDF + SVM

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score
from scipy.sparse import hstack

In [ ]:
train = pd.read_csv('./train.csv')
test = pd.read_csv('./test.csv')

In [ ]:
# Data Preprocessing
vec_facts = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
vec_party = TfidfVectorizer(max_features=2000)

def get_vector(df, train_mode):
    if train_mode:
        X_facts = vec_facts.fit_transform(df['facts'])
        vec_party.fit(pd.concat([df['first_party'], df['second_party']]))
    else:
        X_facts = vec_facts.transform(df['facts'])
    X_p1 = vec_party.transform(df['first_party'])
    X_p2 = vec_party.transform(df['second_party'])
    return hstack([X_p1, X_p2, X_facts]).tocsr()

X = get_vector(train, True)
y = train["first_party_winner"].values
X_test = get_vector(test, False)

In [ ]:
# Base Model: 5-fold CV + LightGBM
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_pred = np.zeros(len(train))
test_pred_sum = np.zeros(len(test))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X[tr_idx], X[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    svc = LinearSVC(C=0.02, max_iter=5000)
    svc.fit(X_tr, y_tr)

    va_pred = svc.predict(X_va)
    oof_pred[va_idx] = va_pred
    test_pred_sum += svc.decision_function(X_test)  # score 누적 (부호로 최종 결정)

    acc = accuracy_score(y_va, va_pred)
    f1 = f1_score(y_va, va_pred, average="macro")
    print(f"Fold {fold+1} | Acc: {acc:.4f} | Macro F1: {f1:.4f}")

print("\nOverall OOF Accuracy:", accuracy_score(y, oof_pred))
print("Overall OOF Macro F1:", f1_score(y, oof_pred, average="macro"))


Fold 1 | Acc: 0.6270 | Macro F1: 0.5588
Fold 2 | Acc: 0.5927 | Macro F1: 0.5210
Fold 3 | Acc: 0.6210 | Macro F1: 0.5678
Fold 4 | Acc: 0.6444 | Macro F1: 0.5923
Fold 5 | Acc: 0.5899 | Macro F1: 0.5274

Overall OOF Accuracy: 0.6150121065375302
Overall OOF Macro F1: 0.553771405273172


In [ ]:
submit = pd.read_csv('./sample_submission.csv')
submit['first_party_winner'] = (test_pred_sum > 0).astype(int)
submit.to_csv('./baseline-svm_submit.csv', index=False)
print('Done')

Done


Baseline Result = 0.5080645161